# Google Capstone Project

# Importing Gemini API Key

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv 

load_dotenv() 
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

if not GOOGLE_API_KEY:
    raise ValueError("API key not found. Please set the GOOGLE_API_KEY in your .env file.")


genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API Key loaded and configured successfully.")


Gemini API Key loaded and configured successfully.


# Using RAG in my documents


In [16]:
# --------------------------------------------------------------------------
# 1. IMPORTS
# --------------------------------------------------------------------------
import os
import google.generativeai as genai
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.generativeai import types
from dotenv import load_dotenv
# --- Ensure these imports are present near the top of your file ---

# Import the protos module
from google.generativeai import protos 
# Import for displaying markdown in Jupyter environments (like VS Code notebooks)
from IPython.display import Markdown, display

# --------------------------------------------------------------------------
# 2. API KEY SETUP (using .env file)
# --------------------------------------------------------------------------
load_dotenv() # Load variables from the .env file

GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    raise ValueError("API key not found. Please set the GOOGLE_API_KEY in your .env file.")

genai.configure(api_key=GOOGLE_API_KEY)
print("Gemini API Key loaded and configured successfully.")

# --------------------------------------------------------------------------
# 3. LOAD WORKOUT LOG DOCUMENTS FROM TXT FILES
# --------------------------------------------------------------------------
import os # Ensure os is imported if not already

document_files = [
    "Jan2025.txt",
    "Jun2024.txt",
    "March2025.txt",
    "WeeklyWorkout.txt"
]
documents = [] # List to hold the content strings
doc_ids = []   # List to hold the document IDs

print(f"Attempting to load {len(document_files)} documents from .txt files...")

# Loop through the filenames, read each file, and store content
for i, filename in enumerate(document_files):
    doc_id = f"doc_{i+1}" # Generate ID like doc_1, doc_2, etc.
    try:
        # Assuming the txt files are in the same directory as the script/notebook
        # Use utf-8 encoding for broader character support
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read()
            documents.append(content)
            doc_ids.append(doc_id)
            print(f"Successfully loaded '{filename}' (ID: {doc_id})")
    except FileNotFoundError:
        print(f"ERROR: File not found - '{filename}'. Please ensure it's in the correct directory.")
        # Optional: Append a placeholder or skip if a file is missing
        # documents.append(f"Content for {filename} not found.")
        # doc_ids.append(doc_id)
    except Exception as e:
        print(f"ERROR: Could not read file '{filename}'. Reason: {e}")
        # Optional: Append a placeholder or skip

# Ensure all documents were loaded successfully before proceeding if necessary
if len(documents) != len(document_files):
    print("\nWarning: Not all documents were loaded successfully. Subsequent steps might fail.")
else:
    print(f"\nSuccessfully loaded content for {len(documents)} documents into the 'documents' list.")



# --------------------------------------------------------------------------
# 4. DEFINE GEMINI EMBEDDING FUNCTION FOR CHROMADB
# --------------------------------------------------------------------------
class GeminiEmbeddingFunction(EmbeddingFunction):
    """Custom embedding function using the Gemini API"""
    def __init__(self, model_name="models/text-embedding-004", task_type="retrieval_document"):
        self.model_name = model_name
        self.task_type = task_type

    def __call__(self, input: Documents) -> Embeddings:
        # Ensure task_type is set correctly for the call
        response = genai.embed_content(
            model=self.model_name,
            content=input,
            task_type=self.task_type
        )
        return response['embedding']

# --------------------------------------------------------------------------
# 5. SETUP CHROMADB & ADD DOCUMENTS
# --------------------------------------------------------------------------
print("\n--- Setting up ChromaDB ---")
# Use a persistent client if you want the database to save to disk
# chroma_client = chromadb.PersistentClient(path="./chroma_db") # Choose a path
chroma_client = chromadb.Client() # In-memory client (cleared when script stops)

db_collection_name = "fitness_logs" # Use a new name or manage clearing old one

# Create the embedding function instance for storing documents
embed_fn_docs = GeminiEmbeddingFunction(task_type="retrieval_document")

# Get or create the collection
db = chroma_client.get_or_create_collection(
    name=db_collection_name,
    embedding_function=embed_fn_docs # Use the doc embedding function here
)

# Add the documents to the collection
# Note: If the collection already exists and has these IDs, add might behave differently (update/ignore).
# Consider db.upsert() if you want to add or update.
try:
    db.add(documents=documents, ids=doc_ids)
    print(f"Successfully added/updated {len(documents)} documents.")
    print(f"Total documents in collection: {db.count()}")
except Exception as e:
    print(f"Error adding documents (they might already exist?): {e}")
    print(f"Document count remains: {db.count()}")

# --------------------------------------------------------------------------
# 6. RAG (RETRIEVAL-AUGMENTED GENERATION) EXAMPLE
# --------------------------------------------------------------------------
print("\n--- RAG Example ---")
user_query_rag = "How many times did I run 10km the last year?"
print(f"User Query: {user_query_rag}")

# Create a separate embedding function instance configured for queries
embed_fn_query = GeminiEmbeddingFunction(task_type="retrieval_query")

# Query ChromaDB
query_results = db.query(
    query_texts=[user_query_rag],
    n_results=3, # Retrieve top 3 most relevant document chunks
    include=['documents'] # We need the document text
    # Let ChromaDB use the collection's default function for the query embedding OR
    # explicitly pass the query embedder if needed/supported by your Chroma version
    # embedding_function=embed_fn_query # Try uncommenting if query doesn't work well
)

# Extract context
retrieved_docs = query_results.get('documents', [[]])[0] # Safer extraction
if not retrieved_docs:
    print("Could not retrieve relevant documents for the query.")
    rag_context = "No relevant documents found."
else:
    rag_context = "\n\n---\n\n".join(retrieved_docs)
    print(f"\nRetrieved Context:\n{rag_context[:500]}...") # Print start of context

# Prepare prompt for the LLM
rag_prompt = f"""You are a helpful fitness assistant analyzing workout logs. Answer the user's question based *only* on the provided context. If the context doesn't contain the answer, say so.

Context from workout logs:
---
{rag_context}
---

User Question: {user_query_rag}

Answer:"""

# Generate the answer using a Gemini model
rag_model = genai.GenerativeModel('gemini-1.5-flash') # Or 'gemini-pro'
try:
    rag_response = rag_model.generate_content(rag_prompt)
    print("\nGenerated RAG Answer:")
    # Use display(Markdown(...)) if in a Jupyter environment
    # display(Markdown(rag_response.text))
    # Use print() if in a standard Python script
    print(rag_response.text)
except Exception as e:
    print(f"Error generating RAG response: {e}")




Gemini API Key loaded and configured successfully.
Attempting to load 4 documents from .txt files...
Successfully loaded 'Jan2025.txt' (ID: doc_1)
Successfully loaded 'Jun2024.txt' (ID: doc_2)
Successfully loaded 'March2025.txt' (ID: doc_3)
Successfully loaded 'WeeklyWorkout.txt' (ID: doc_4)

Successfully loaded content for 4 documents into the 'documents' list.

--- Setting up ChromaDB ---
Successfully added/updated 4 documents.
Total documents in collection: 4

--- RAG Example ---
User Query: How many times did I run 10km the last year?

Retrieved Context:
Log workout 2
25-mar-2025

Chest 
Bike zone 2


Oatmeal cookies
Salmon 
Etc

26-mar-2025

Back
Cardio

27-mar-2025

Oatmeal + pineapple
3 boiled eggs
Some avocado


28-march-2024

Leg workout

Oatmeal paw paw

3 egg whites
Ham slices
Avocado
1 small donut
Cappuccino decaf

29-mar-2025

Shoulders
Jumping rope and boxing

76kgs

30-mar-2025

10 km run

31-mar-2025

Chest

Meals
Oatmeal cookies, protein shake with banana, three boiled

In [ ]:
# --------------------------------------------------------------------------
# 7. STRUCTURED OUTPUT (JSON MODE) 
# --------------------------------------------------------------------------
import json # Import json library for parsing/pretty-printing

print("\n--- Structured Output (JSON) Example ---")

# Define which document from the list we want to analyze by its index
# Index 1 corresponds to the second file in the list: "Jun2024.txt"
target_doc_index = 1

# Check if the target document exists in the loaded list
if target_doc_index < len(documents):
    target_doc_content = documents[target_doc_index]
    target_doc_name = document_files[target_doc_index] # Get the filename for context

    target_date = "19-August-2024" # The date we want to extract from the target document
    print(f"Target Document: '{target_doc_name}' (documents[{target_doc_index}])")
    print(f"Target Date for Extraction: {target_date}")

    # --- Prepare Prompt for JSON Output ---
    # Instruct the model to analyze the specific document content provided
    json_prompt = f"""
Analyze the following workout log document content, which comes from the file '{target_doc_name}'.
Find the entry specifically for the date "{target_date}".
Extract the main workout activities performed and the meals listed for that day.
Return the result ONLY as a valid JSON object with the following keys:
- "date" (string: the target date found, or null)
- "workout_summary" (string: a brief summary of workouts, or null)
- "meals" (list of strings: each meal listed for that day, or empty list [])

If the date cannot be found or information is missing for that date, return appropriate null or empty values within the JSON structure, but still return a valid JSON object.

Document Content from '{target_doc_name}':
---
{target_doc_content}
---

JSON Output:
"""

    # --- Configure for JSON Output ---
    json_generation_config = genai.types.GenerationConfig(
        response_mime_type='application/json',
    )

    # --- Generate JSON using Gemini ---
    print("\nGenerating Structured (JSON) Output...")
    try:
        json_model = genai.GenerativeModel('gemini-1.5-flash') # Using gemini-pro
        json_response = json_model.generate_content(
            contents=json_prompt,
            generation_config=json_generation_config,
        )

        print("\nGenerated JSON Output:")
        # The response text should be valid JSON
        print(json_response.text)

        # Optional: Parse and pretty-print the JSON
        try:
            parsed_json = json.loads(json_response.text)
            print("\nParsed JSON:")
            print(json.dumps(parsed_json, indent=2))
        except json.JSONDecodeError:
            print("\nWarning: Could not parse the output as valid JSON.")

    except Exception as e:
        print(f"Error generating JSON response: {e}")

else:
    print(f"ERROR: Cannot run Section 7. Document index {target_doc_index} is out of bounds for the loaded documents list (length {len(documents)}).")
    print("Please ensure the document files were loaded correctly in Section 3.")


print("\n--- Structured Output Section Finished ---")


--- Structured Output (JSON) Example ---
Target Document: 'Jun2024.txt' (documents[1])
Target Date for Extraction: 19-August-2024

Generating Structured (JSON) Output...

Generated JSON Output:
{"date": "19-August-2024", "workout_summary": "Chest workout and swimming workout at night", "meals": ["Oatmeal w paw paw", "3 boiled white eggs w 2 turkey slices w panela cheese", "Chicken w Rice", "Apple", "Chicken w Rice", "Protein shake"]}

Parsed JSON:
{
  "date": "19-August-2024",
  "workout_summary": "Chest workout and swimming workout at night",
  "meals": [
    "Oatmeal w paw paw",
    "3 boiled white eggs w 2 turkey slices w panela cheese",
    "Chicken w Rice",
    "Apple",
    "Chicken w Rice",
    "Protein shake"
  ]
}

--- Structured Output Section Finished ---
